In [ ]:
!git clone https://github.com/bababyVN/ProjCv.git
%cd ProjCv

!pip install -r requirements.txt

import sys
from pathlib import Path

# Load config and override values for Kaggle environment
config_path = Path("config.py")
content = config_path.read_text(encoding="utf-8")

# Modify dataset paths to point to Kaggle input directory
content = content.replace(
    '"DATA_ROOT":     str(_PROJECT_ROOT / "data"),',
    '"DATA_ROOT":     "/kaggle/input/deepglobe-land-cover-classification-dataset",'
)

# Modify output directory to Kaggle working directory
content = content.replace(
    '"OUTPUT_DIR":    str(_PROJECT_ROOT / "output"),',
    '"OUTPUT_DIR":    "/kaggle/working/output",'
)

content = content.replace(
    '"BEST_MODEL":    str(_PROJECT_ROOT / "output" / "best_model.pth"),',
    '"BEST_MODEL":    "/kaggle/working/output/best_model.pth",'
)

# Set suitable hyperparameters for Kaggle GPUs
# (Kaggle allows up to 2 workers for T4, 4 for P100. Batch size 8 is stable for Swin-T)
content = content.replace('"BATCH_SIZE":    8,', '"BATCH_SIZE":    8,')
content = content.replace('"NUM_WORKERS":   2,', '"NUM_WORKERS":   2,')

config_path.write_text(content, encoding="utf-8")
print("Kaggle paths updated in config.py!")

import os
from pathlib import Path
# ── 1. Find the dataset root automatically ────────────────────
dataset_root = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "train" in dirs:
        train_path = Path(root) / "train"
        # Verify it contains satellite images
        sat_files = list(train_path.glob("*_sat.jpg"))
        if len(sat_files) > 0:
            dataset_root = root
            break
if not dataset_root:
    print("ERROR: Could not find the 'train' folder with '_sat.jpg' files under /kaggle/input.")
    print("Please make sure you have added the 'deepglobe-land-cover-classification-dataset' input.")
else:
    print(f"Found dataset root at: {dataset_root}")
    
    # ── 2. Update config.py with the correct path ──────────────
    config_path = Path("config.py")
    content = config_path.read_text(encoding="utf-8")
    # Replace DATA_ROOT with the auto-detected path
    # First, revert any previous replacements
    content = content.replace(
        '"DATA_ROOT":     "/kaggle/input/deepglobe-land-cover-classification-dataset",',
        '"DATA_ROOT":     str(_PROJECT_ROOT / "data"),'
    )
    
    # Apply new replacement
    content = content.replace(
        '"DATA_ROOT":     str(_PROJECT_ROOT / "data"),',
        f'"DATA_ROOT":     "{dataset_root}",'
    )
    # Replace OUTPUT_DIR
    content = content.replace(
        '"OUTPUT_DIR":    str(_PROJECT_ROOT / "output"),',
        '"OUTPUT_DIR":    "/kaggle/working/output",'
    )
    # Replace BEST_MODEL
    content = content.replace(
        '"BEST_MODEL":    str(_PROJECT_ROOT / "output" / "best_model.pth"),',
        '"BEST_MODEL":    "/kaggle/working/output/best_model.pth",'
    )
    config_path.write_text(content, encoding="utf-8")
    print("config.py successfully updated!")

!python scripts/train.py

!zip -r model_output.zip /kaggle/working/output
